# Sentinel-1 Acquisition for RoughNet

This notebook queries the Copernicus Data Space Ecosystem (CDSE) for Sentinel-1 GRD products covering the existing LiDAR study patches, selects scenes close to each LiDAR acquisition date, downloads the SAFE archives, and extracts them.

Credentials are read from environment variables. Do not place a password or access token in this notebook. This notebook is intentionally unexecuted; run it on the GPU machine only when the data paths and credentials are configured.

## Configuration

The AOI is built from existing `lidar_patch_*.tif` files. The resulting products are saved under `raw_data/<region>_sentinel1_downloads/`.

In [21]:
import os
import time
import json
import zipfile
import datetime as dt
import shapely.wkt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.warp import transform_geom
from shapely.geometry import box, shape
from shapely.ops import unary_union

from pyproj import Transformer
from shapely.ops import transform as shp_transform


In [34]:
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'tuk'  # change to pondinlet or cambridge
LIDAR_DIR = INPUT_DIR / f'lidar_patches_{REGION}'
RAW_DIR = REPO_DIR / 'raw_data' / f'{REGION}_sentinel1_downloads'
DATE_BY_REGION = {
    'pondinlet': dt.date(2024, 4, 26),
    'cambridge': dt.date(2024, 4, 18),
    'tuk': dt.date(2024, 4, 16),
}
SEARCH_DAYS = 30
MAX_PRODUCTS = 8
PRODUCT_TYPES = ['IW_GRDH_1S']
CDSE_USERNAME = os.environ['CDSE_USERNAME']
CDSE_PASSWORD = os.environ['CDSE_PASSWORD']
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Build the search AOI

Sampling the patch bounds keeps the CDSE query geometry compact while still covering the LiDAR study area.

In [23]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
AOI_WKT = aoi.convex_hull.wkt   # tighter than envelope, still short enough for the query URL
if aoi.geom_type == 'MultiPolygon':
    n_vertices = sum(len(poly.exterior.coords) for poly in aoi.geoms)
else:
    n_vertices = len(aoi.exterior.coords)
print(f'AOI vertices (full): {n_vertices}')
print(f'AOI_WKT (convex hull, for query): {AOI_WKT}')




AOI vertices (full): 1024
AOI_WKT (convex hull, for query): POLYGON ((-133.33597812407788 69.69784642903053, -133.34044267002972 69.69954197956464, -133.35778073644605 69.71291381705595, -133.3598837050864 69.71540900315729, -133.36264387428858 69.73852376644697, -133.35841894347007 69.79124640264024, -133.35735930270482 69.80442631910744, -133.35629829343503 69.81760593892227, -133.35365359903727 69.8217008507903, -133.2887905368664 69.84987776213973, -133.28401985003077 69.85147576501868, -133.27952671544594 69.84977906989435, -133.2774197210839 69.84728338997641, -133.27641183388263 69.80440288270277, -133.27667344233464 69.77390037038805, -133.27887268200058 69.7203298530142, -133.2790109730542 69.71868221441274, -133.32651402151177 69.70104589727221, -133.33597812407788 69.69784642903053))


## Authenticate and query CDSE

In [30]:
TOKEN_URL = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
CATALOGUE_URL = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'

def get_access_token(username, password):
    response = requests.post(TOKEN_URL, data={
        'grant_type': 'password',
        'username': username,
        'password': password,
        'client_id': 'cdse-public',
    }, timeout=60)
    response.raise_for_status()
    return response.json()['access_token']

access_token = get_access_token(CDSE_USERNAME, CDSE_PASSWORD)
headers = {'Authorization': f'Bearer {access_token}'}
start = DATE_BY_REGION[REGION] - dt.timedelta(days=SEARCH_DAYS)
end = DATE_BY_REGION[REGION] + dt.timedelta(days=SEARCH_DAYS)
product_filter = ' or '.join([
    f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq '{kind}')"
    for kind in PRODUCT_TYPES
])
odata_filter = (
    "Collection/Name eq 'SENTINEL-1' and "
    f"OData.CSC.Intersects(area=geography'SRID=4326;{AOI_WKT}') and "
    f"ContentDate/Start ge {start.isoformat()}T00:00:00.000Z and "
    f"ContentDate/Start le {end.isoformat()}T23:59:59.999Z and "
    f"({product_filter})"
)
params = {'$filter': odata_filter, '$top': 200, '$orderby': 'ContentDate/Start asc'}
response = requests.get(CATALOGUE_URL, params=params, headers=headers, timeout=120)
response.raise_for_status()
products = pd.DataFrame(response.json().get('value', []))
print(f'Catalogue products returned: {len(products)}')

Catalogue products returned: 8


In [31]:
def filter_by_coverage(df_results, aoi_ll, threshold=0.90):
    """Keep only products covering >= threshold fraction of the AOI (or fully covering it)."""
    to_eq = Transformer.from_crs('EPSG:4326', 'EPSG:6933', always_xy=True)
    aoi_eq = shp_transform(lambda x, y: to_eq.transform(x, y), aoi.buffer(0))
    aoi_area = aoi_eq.area if aoi_eq.area > 0 else 1.0

    def parse_footprint(row):
        w = row.get('Footprint')
        if not w or not isinstance(w, str):
            return None
        s = w.strip()
        if s.lower().startswith('geography'):
            first, last = s.find("'"), s.rfind("'")
            if first != -1 and last != -1 and last > first:
                s = s[first + 1:last]
        if s.upper().startswith('SRID='):
            s = s.split(';', 1)[-1].strip()
        try:
            g = shapely.wkt.loads(s)
            return g.buffer(0) if not g.is_valid else g
        except Exception:
            return None

    fracs, covers = [], []
    for _, r in df_results.iterrows():
        g = parse_footprint(r)
        if g is None:
            fracs.append(0.0); covers.append(False)
            continue
        prod_eq = shp_transform(lambda x, y: to_eq.transform(x, y), g.buffer(0))
        frac = float(prod_eq.intersection(aoi_eq).area / aoi_area)
        fracs.append(frac)
        covers.append(bool(g.buffer(0).covers(aoi.buffer(0))))

    out = df_results.copy()
    out['CoverageFracAOI'] = fracs
    out['CoversAOI'] = covers
    return out[(out['CoversAOI']) | (out['CoverageFracAOI'] >= threshold)].copy()

products = filter_by_coverage(products, aoi, threshold=0.90)
print(f'Products after coverage filter: {len(products)}')


Products after coverage filter: 8


In [33]:
print(products[['Id', 'Name', 'CoverageFracAOI', 'CoversAOI']].to_string())


                                     Id                                                                      Name  CoverageFracAOI  CoversAOI
0  ade07c7d-dad8-4688-a7cb-451a675ddae0  S1A_IW_GRDH_1SDV_20240318T022516_20240318T022540_053030_066BC8_843D.SAFE              1.0       True
1  3b576c7e-2950-4da1-90ae-bcf9e759a4c2  S1A_IW_GRDH_1SDV_20240320T020855_20240320T020919_053059_066CEB_40A4.SAFE              1.0       True
2  f664e9d5-a50b-46dc-abdc-2675bb7083c6  S1A_IW_GRDH_1SDV_20240401T020855_20240401T020919_053234_067390_6689.SAFE              1.0       True
3  f8842a88-8d0f-4e17-baeb-b63c2911a292  S1A_IW_GRDH_1SDV_20240413T020854_20240413T020918_053409_067A75_8A2D.SAFE              1.0       True
4  82adaffe-9f6e-4548-8ea8-228f0ba5b25b  S1A_IW_GRDH_1SDV_20240423T022517_20240423T022541_053555_068037_E833.SAFE              1.0       True
5  ed9ce6bc-c67d-4de1-a329-38299e13c29f  S1A_IW_GRDH_1SDV_20240425T020855_20240425T020920_053584_068151_25F8.SAFE              1.0       True
6  9e6

## Select, download, and extract products

Selection is based on acquisition date proximity. Inspect the resulting table before downloading at scale.

In [35]:
if products.empty:
    raise RuntimeError('No Sentinel-1 products matched the query.')
products['acquisition_date'] = pd.to_datetime(products['ContentDate'].map(lambda x: x['Start'])).dt.date
products['date_distance_days'] = products['acquisition_date'].map(lambda x: abs((x - DATE_BY_REGION[REGION]).days))
selected = products.sort_values(['date_distance_days', 'acquisition_date']).head(MAX_PRODUCTS).copy()
display(selected[['Id', 'Name', 'acquisition_date', 'date_distance_days']])

def download_product(row, output_dir, token):
    product_id = row['Id']
    name = row['Name']
    zip_path = output_dir / f'{name}.zip'
    if zip_path.exists() and zip_path.stat().st_size > 0:
        return zip_path
    url = f'https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value'
    with requests.get(url, headers={'Authorization': f'Bearer {token}'}, stream=True, timeout=300) as r:
        r.raise_for_status()
        with zip_path.open('wb') as handle:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return zip_path

downloaded = []
for _, row in selected.iterrows():
    access_token = get_access_token(CDSE_USERNAME, CDSE_PASSWORD)
    path = download_product(row, RAW_DIR, access_token)
    downloaded.append(path)
    print(path.name, path.stat().st_size, 'bytes')

for zip_path in downloaded:
    extract_dir = RAW_DIR / zip_path.stem
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)
    print('Extracted:', extract_dir)

,Id,Name,acquisition_date,date_distance_days
3,f8842a88-8d0f-4e17-baeb-b63c2911a292,S1A_IW_GRDH_1SDV_20240413T020854_20240413T0209...,2024-04-13,3
4,82adaffe-9f6e-4548-8ea8-228f0ba5b25b,S1A_IW_GRDH_1SDV_20240423T022517_20240423T0225...,2024-04-23,7
5,ed9ce6bc-c67d-4de1-a329-38299e13c29f,S1A_IW_GRDH_1SDV_20240425T020855_20240425T0209...,2024-04-25,9
2,f664e9d5-a50b-46dc-abdc-2675bb7083c6,S1A_IW_GRDH_1SDV_20240401T020855_20240401T0209...,2024-04-01,15
6,9e6e3ba2-dd9d-41ff-87af-0b3e9925efbd,S1A_IW_GRDH_1SDV_20240505T022517_20240505T0225...,2024-05-05,19
7,7c37b2f8-debe-49dc-b9f7-4f50d92884df,S1A_IW_GRDH_1SDV_20240507T020856_20240507T0209...,2024-05-07,21
1,3b576c7e-2950-4da1-90ae-bcf9e759a4c2,S1A_IW_GRDH_1SDV_20240320T020855_20240320T0209...,2024-03-20,27
0,ade07c7d-dad8-4688-a7cb-451a675ddae0,S1A_IW_GRDH_1SDV_20240318T022516_20240318T0225...,2024-03-18,29


S1A_IW_GRDH_1SDV_20240413T020854_20240413T020918_053409_067A75_8A2D.SAFE.zip 1718980773 bytes
S1A_IW_GRDH_1SDV_20240423T022517_20240423T022541_053555_068037_E833.SAFE.zip 1736632988 bytes
S1A_IW_GRDH_1SDV_20240425T020855_20240425T020920_053584_068151_25F8.SAFE.zip 1718975012 bytes
S1A_IW_GRDH_1SDV_20240401T020855_20240401T020919_053234_067390_6689.SAFE.zip 1720006142 bytes
S1A_IW_GRDH_1SDV_20240505T022517_20240505T022542_053730_06871A_EE55.SAFE.zip 1735861383 bytes
S1A_IW_GRDH_1SDV_20240507T020856_20240507T020920_053759_06883A_0ED4.SAFE.zip 1717685940 bytes
S1A_IW_GRDH_1SDV_20240320T020855_20240320T020919_053059_066CEB_40A4.SAFE.zip 1720135773 bytes
S1A_IW_GRDH_1SDV_20240318T022516_20240318T022540_053030_066BC8_843D.SAFE.zip 1735496764 bytes
Extracted: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/tuk_sentinel1_downloads/S1A_IW_GRDH_1SDV_20240413T020854_20240413T020918_053409_067A75_8A2D.SAFE
Extracted: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/tuk_sentinel1_downloads/S

## Output contract

The next notebook expects one or more extracted `*.SAFE` directories under `RAW_DIR`. It reads VV/VH measurement TIFFs and their calibration XML files from those SAFE products.